# <span style="color:blue">PART 4: INTEGRATED DATA PIPELINE</span> ⏱️ 

---
In this part, we **integrate everything** learned so far — databases, APIs, and web scraping — into a **unified, production-style data pipeline**.

> 💡 **Goal:** Build a reusable `DataCollectionPipeline` class that pulls data from multiple sources, stores it in SQLite, logs every step, and exports results.

In [83]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from urllib.parse import urljoin
import os
import logging
import sqlite3

## <span style="color:blue">4.1 Complete Data Collection System</span>

Now let's **integrate everything**: databases, APIs, and web scraping!

The `DataCollectionPipeline` class provides a **unified interface** for:
- 🗄️ Querying **SQLite databases**
- 🌐 Fetching data from **REST APIs**
- 🕷️ **Web scraping** with BeautifulSoup
- 📝 **Logging** every operation to both file and console
- 📊 **Exporting** all collected data to CSV files

### Imports Overview

| Library | Purpose |
|---|---|
| `sqlite3` | Database connectivity |
| `requests` | HTTP requests for APIs and web scraping |
| `BeautifulSoup` | HTML parsing |
| `pandas` | Data manipulation and export |
| `logging` | Structured logging |
| `datetime`, `time` | Timestamps and rate limiting |
| `json` | Serializing API responses for DB storage |

In [84]:
# ─── Core standard library imports ───────────────────────────────────────────
import sqlite3
import logging
from datetime import datetime
import time
import json
import os
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

# ─── Third-party imports (install via pip if needed) ─────────────────────────
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import pandas as pd

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


### <span style="color:blue">`DataCollectionPipeline` Class</span>

The class is organized into four method groups:
1. **`__init__` & `_create_tables`** — Setup: logging, DB connection, tables
2. **Database methods** — `collect_from_database()`
3. **API methods** — `collect_from_api()`
4. **Web scraping methods** — `collect_from_web()`
5. **Utility methods** — `_log_collection()`, `get_collection_stats()`, `export_all_data()`, `close()`

> <span style="color:red">**Important:**</span> The pipeline uses a **single SQLite database** (`collected_data.db` by default) to store data from **all sources**, keeping everything in one place for easy analysis.

In [85]:
class RateLimiter:

    def __init__(self, max_requests=60, time_window=3600):
        self.max_requests = max_requests
        self.time_window = time_window
        self.requests = []  

    def wait_if_needed(self):

        now = time.time()
        
        self.requests = [t for t in self.requests if now - t < self.time_window]
        
        if len(self.requests) >= self.max_requests:
            sleep_time = self.time_window - (now - self.requests[0])
            if sleep_time > 0:
                print(f"⏰ Rate limit reached. Sleeping for {sleep_time:.1f} seconds...")
                time.sleep(sleep_time)
            self.requests = []

        self.requests.append(now)


class DataCollectionPipeline:
    """
    Unified data collection from multiple sources.
    Integrates: SQLite databases, REST APIs, and Web Scraping.
    """

    # =========================================================================
    # INITIALIZATION
    # =========================================================================

    def __init__(self, db_path='collected_data.db'):
        """
        Initialize pipeline with database and logging.

        Args:
            db_path (str): Path to the SQLite database file.
                           Will be created if it doesn't exist.
        """
        # ── Setup logging ─────────────────────────────────────────────────────
        # Logs go to BOTH a file ('pipeline.log') and the console (StreamHandler)
        logs_dir = "logs"
        os.makedirs(logs_dir, exist_ok=True)
        logs_path = os.path.join(logs_dir, "pipeline.log")

        logging.basicConfig(
            level=logging.INFO,  # Log INFO and above (INFO, WARNING, ERROR, CRITICAL)
            format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
            handlers=[
                logging.FileHandler(logs_path),  # Persistent log file
                logging.StreamHandler(),  # Real-time console output
            ],
        )
        self.logger = logging.getLogger(__name__)  # Logger named after this module

        # ── Setup database ────────────────────────────────────────────────────
        db_dir = "databases"
        os.makedirs(db_dir, exist_ok=True)
        self.db_path = os.path.join(db_dir, db_path)
        self.conn = sqlite3.connect(self.db_path)  # Creates file if it doesn't exist
        self._create_tables()  # Ensure all tables exist

        # ── Setup HTTP session ──────────────────────────────────────────────── 
        self.session = requests.Session()
        retry_strategy = Retry(
            total=3,                                     
            backoff_factor=1,                            
            status_forcelist=[429, 500, 502, 503, 504], 
            allowed_methods=['HEAD', 'GET', 'POST', 'PUT', 'DELETE', 'OPTIONS'],
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        self.session.mount('http://', adapter)
        self.session.mount('https://', adapter)
        self.session.headers.update(
            {'User-Agent': 'Asmaa Bot'} 
        )

        self.rate_limiter = RateLimiter(max_requests=60, time_window=3600)

        self.logger.info("Pipeline initialized")

    def _create_tables(self):
        """
        Create all required tables if they don't already exist.
        Uses CREATE TABLE IF NOT EXISTS so it's safe to call multiple times.
        """
        cursor = self.conn.cursor()

        # ── Table for API data ────────────────────────────────────────────────
        # Stores the full JSON response as TEXT (json.dumps)
        cursor.execute(
            '''
            CREATE TABLE IF NOT EXISTS api_data (
                id           INTEGER PRIMARY KEY AUTOINCREMENT,
                source       TEXT NOT NULL,       -- URL of the API endpoint
                data_type    TEXT NOT NULL,       -- e.g. 'json'
                content      TEXT NOT NULL,       -- JSON-serialized response
                collected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        '''
        )

        # ── Table for scraped data ────────────────────────────────────────────
        # Stores the URL, page title, and JSON-encoded field dict
        cursor.execute(
            '''
            CREATE TABLE IF NOT EXISTS scraped_data (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                url        TEXT NOT NULL,
                title      TEXT,
                content    TEXT,          -- JSON-encoded dict of scraped fields
                scraped_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        '''
        )

        # ── Table for pipeline operation logs ─────────────────────────────────
        # Audit trail: which source, how many records, success/error, and why
        cursor.execute(
            '''
            CREATE TABLE IF NOT EXISTS pipeline_logs (
                id               INTEGER PRIMARY KEY AUTOINCREMENT,
                source_type      TEXT NOT NULL,       -- 'database', 'api', or 'web'
                records_collected INTEGER,
                status           TEXT,               -- 'success' or 'error'
                error_message    TEXT,               -- NULL on success
                timestamp        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        '''
        )

        self.conn.commit()  # Persist table creation

    # =========================================================================
    # DATABASE METHODS
    # =========================================================================

    def collect_from_database(self, query, source_db_path):
        """
        Collect data from another SQLite database by running a SQL query.

        Args:
            query (str):           SQL SELECT query to run on the source DB.
            source_db_path (str):  Path to the source .db file.

        Returns:
            pd.DataFrame: Query results. Empty DataFrame on error.
        """
        self.logger.info(f"Collecting from database: {source_db_path}")
        try:
            # Open a separate connection to the source database
            source_conn = sqlite3.connect(source_db_path)

            # pandas read_sql_query runs the query and returns a DataFrame directly
            df = pd.read_sql_query(query, source_conn)
            source_conn.close()  # Always close the connection when done

            self.logger.info(f"✓ Collected {len(df)} records from database")
            self._log_collection('database', len(df), 'success')  # Audit log
            return df

        except Exception as e:
            # Catch-all: file not found, SQL syntax error, permission issues, etc.
            self.logger.error(f"✗ Database error: {e}")
            self._log_collection('database', 0, 'error', str(e))
            return pd.DataFrame()  # Return empty DF so callers don't need to check None

    # =========================================================================
    # API METHODS
    # =========================================================================

    def _check_rate_limit_header(self, response):

        if 'X-RateLimit-Remaining' in response.headers:
            remaining = int(response.headers['X-RateLimit-Remaining'])
            limit     = int(response.headers.get('X-RateLimit-Limit', 0))
            reset_ts  = int(response.headers.get('X-RateLimit-Reset', 0))
            reset_dt  = datetime.fromtimestamp(reset_ts) if reset_ts else 'unknown'
            self.logger.info(f"Rate limit: {remaining}/{limit} — resets at {reset_dt}")
            if remaining < 10:
                self.logger.warning("⚠️ Low on API requests!")


    def collect_from_api(self, url, params=None):
        """
        Collect JSON data from a REST API endpoint via GET request.
        Retries are handled by the session's HTTPAdapter.
        """

        self.logger.info(f"Collecting from API: {url}")

        try:
            self.rate_limiter.wait_if_needed()

            response = self.session.get(url, params=params, timeout=10)

            if response.status_code == 200:
                self._check_rate_limit_header(response)

                data = response.json()

                cursor = self.conn.cursor()
                cursor.execute(
                    '''
                    INSERT INTO api_data (source, data_type, content)
                    VALUES (?, ?, ?)
                    ''',
                    (url, "json", json.dumps(data)),
                )
                self.conn.commit()

                self.logger.info(f"✓ Collected API data from {url}")
                self._log_collection("api", 1, "success")

                return data

            elif response.status_code == 404:
                self.logger.error(f"✗ Not found: {url}")
                self._log_collection("api", 0, "error", "404 Not Found")
                return None

            else:
                response.raise_for_status()

        except requests.exceptions.Timeout:
            self.logger.error("Request timed out")
            self._log_collection("api", 0, "error", "Timeout")

        except requests.exceptions.ConnectionError as e:
            self.logger.error(f"Connection error: {e}")
            self._log_collection("api", 0, "error", str(e))

        except ValueError:
            self.logger.error("Invalid JSON response")
            self._log_collection("api", 0, "error", "Invalid JSON")

        except Exception as e:
            self.logger.error(f"✗ API error: {e}")
            self._log_collection("api", 0, "error", str(e))

        return None

    # =========================================================================
    # WEB SCRAPING METHODS
    # =========================================================================

    def _can_scrape(self, url):

        parsed = urlparse(url)
        rp = None

        try:
            rp = RobotFileParser()
            rp.set_url(f"{parsed.scheme}://{parsed.netloc}/robots.txt")
            rp.read()
        except Exception:
            self.logger.warning(f"⚠️ Could not read robots.txt for {domain} — proceeding")

        if rp is None:
            return True  

        allowed = rp.can_fetch('*', url)
        
        if not allowed:
            self.logger.warning(f"❌ robots.txt disallows scraping: {url}")
        
        return allowed

    def collect_from_web(
        self, url, selectors, cleaners=None, validators=None,
        container_selector=None, next_page_selector=None, max_pages=1
    ):
        """
        Scrape structured data from a webpage using CSS selectors.

        Args:
            url (str):                       Starting page URL to scrape.
            selectors (dict):                Mapping of field name → CSS selector string.
            cleaners (dict, optional):       Mapping of field name → callable(Tag | None)
            validators (dict, optional):     Mapping of field name → callable(cleaned_value) → bool. 
            container_selector (str, optional): CSS selector for the repeating item containe 
            next_page_selector (str, optional): CSS selector for the 'next page' link element
            max_pages (int):                 Maximum number of pages to scrape when paginating.

        Returns:
            list[dict] when container_selector is given, dict otherwise.
            Returns [] or {} on error.
        """

        def _extract(scope):
            """Extract one dict of fields from a BS4 scope (soup or container tag)."""
            item = {}
            for field, selector in selectors.items():
                element = scope.select_one(selector)
                if cleaners and field in cleaners:
                    item[field] = cleaners[field](element)  # element may be None
                else:
                    item[field] = element.get_text(strip=True) if element else None
            return item

        def _is_valid(item):
            """Return True only if every validator passes for its field's cleaned value."""
            if not validators:
                return True
            for field, validate in validators.items():
                if not validate(item.get(field)):
                    self.logger.debug(f"Validation failed for field '{field}': {item.get(field)!r}")
                    return False
            return True

        def _scrape_page(page_url):
            """Fetch a single page and return (soup, extracted_items)."""
            self.logger.info(f"Scraping: {page_url}")
            if not self._can_scrape(page_url):
                self._log_collection('web', 0, 'error', 'Blocked by robots.txt')
                return None, []
            response = self.session.get(page_url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'lxml')
            if container_selector:
                items = [_extract(c) for c in soup.select(container_selector)]
            else:
                items = _extract(soup)
            return soup, items

        cursor = self.conn.cursor()

        try:
            if container_selector:
                results = []
                current_url = url
                pages_scraped = 0

                while current_url and (max_pages is None or pages_scraped < max_pages):
                    soup, items = _scrape_page(current_url)
                    if soup is None:  # blocked by robots.txt
                        break

                    accepted = 0
                    for item in items:
                        if not _is_valid(item):  # drop invalid items entirely
                            continue
                        accepted += 1
                        results.append(item)
                        cursor.execute(
                            '''
                            INSERT INTO scraped_data (url, title, content)
                            VALUES (?, ?, ?)
                        ''',
                            (current_url, item.get('title'), json.dumps(item, default=str)),
                        )
                    self.conn.commit()
                    dropped = len(items) - accepted
                    self.logger.info(
                        f"✓ Scraped {len(items)} items from {current_url}"
                        + (f" ({dropped} dropped by validators)" if dropped else "")
                    )
                    pages_scraped += 1

                    # ── Follow next page link if pagination is requested ───────
                    if next_page_selector:
                        nxt = soup.select_one(next_page_selector)
                        current_url = urljoin(current_url, nxt['href']) if nxt else None
                    else:
                        break  # no pagination — stop after first page

                self._log_collection('web', len(results), 'success')
                return results

            else:
                _, data = _scrape_page(url)
                if not data:  # blocked by robots.txt returns []
                    return {}
                if not _is_valid(data):
                    self.logger.warning(f"Item from {url} dropped by validators")
                    self._log_collection('web', 0, 'error', 'Dropped by validators')
                    return {}
                cursor.execute(
                    '''
                    INSERT INTO scraped_data (url, title, content)
                    VALUES (?, ?, ?)
                ''',
                    (url, data.get('title'), json.dumps(data, default=str)),
                )
                self.conn.commit()
                self.logger.info(f"✓ Scraped data from {url}")
                self._log_collection('web', 1, 'success')
                return data

        except Exception as e:
            self.logger.error(f"✗ Scraping error: {e}")
            self._log_collection('web', 0, 'error', str(e))
            return [] if container_selector else {}  # safe empty value for callers

    # =========================================================================
    # UTILITY METHODS
    # =========================================================================

    def _log_collection(self, source_type, records, status, error_msg=None):
        """
        Internal method: Write a collection event to the pipeline_logs table.
        Called automatically after every collect_* method (success or failure).

        Args:
            source_type (str): 'database', 'api', or 'web'
            records (int):     Number of records collected (0 on error)
            status (str):      'success' or 'error'
            error_msg (str):   Exception message if status=='error', else None
        """
        cursor = self.conn.cursor()
        cursor.execute(
            '''
            INSERT INTO pipeline_logs (source_type, records_collected, status, error_message)
            VALUES (?, ?, ?, ?)
        ''',
            (source_type, records, status, error_msg),
        )
        self.conn.commit()

    def get_collection_stats(self):
        """
        Query the database to summarize all data collection activity.

        Returns:
            dict with keys:
                'api_records'     (int): total rows in api_data table
                'scraped_records' (int): total rows in scraped_data table
                'logs'            (DataFrame): per-source success counts
        """
        stats = {}
        cursor = self.conn.cursor()

        # Count total API records collected
        cursor.execute("SELECT COUNT(*) FROM api_data")
        stats['api_records'] = cursor.fetchone()[0]

        # Count total scraped records collected
        cursor.execute("SELECT COUNT(*) FROM scraped_data")
        stats['scraped_records'] = cursor.fetchone()[0]

        # Summarize logs: per source_type — total attempts and how many succeeded
        logs_df = pd.read_sql_query(
            '''
            SELECT
                source_type,
                COUNT(*) AS count,
                SUM(CASE WHEN status='success' THEN 1 ELSE 0 END) AS successful
            FROM pipeline_logs
            GROUP BY source_type
        ''',
            self.conn,
        )
        stats['logs'] = logs_df

        return stats

    def export_all_data(self, output_dir='exports'):
        """
        Export all collected data tables to CSV files.
        Creates the output directory if it doesn't exist.

        Files created:
            {output_dir}/api_data.csv
            {output_dir}/scraped_data.csv
            {output_dir}/pipeline_logs.csv
        """
        os.makedirs(output_dir, exist_ok=True)  # Create dir (no error if exists)

        # Export API data
        api_df = pd.read_sql_query("SELECT * FROM api_data", self.conn)
        api_df.to_csv(f'{output_dir}/api_data.csv', index=False)

        # Export scraped data
        scraped_df = pd.read_sql_query("SELECT * FROM scraped_data", self.conn)
        scraped_df.to_csv(f'{output_dir}/scraped_data.csv', index=False)

        # Export operation logs (useful for debugging and auditing)
        logs_df = pd.read_sql_query("SELECT * FROM pipeline_logs", self.conn)
        logs_df.to_csv(f'{output_dir}/pipeline_logs.csv', index=False)

        self.logger.info(f"✓ Data exported to {output_dir}/")

    def close(self):
        """
        Close the SQLite database connection.
        Always call this when finished to avoid database locking issues.
        """
        self.conn.close()
        self.logger.info("Pipeline closed")


print("✅ DataCollectionPipeline class defined successfully!")

✅ DataCollectionPipeline class defined successfully!


### Usage Example

The following code shows a **complete end-to-end run** of the pipeline:

1. **Collect from database** — query a local SQLite DB (`library.db`)
2. **Collect from API** — fetch a GitHub repository's metadata
3. **Scrape from web** — pull book details from [books.toscrape.com](http://books.toscrape.com)
4. **Get statistics** — see how many records came from each source
5. **Export everything** — save all data to CSV files
6. **Close** — cleanly shut down the pipeline

> <span style="color:red">**Note:**</span> `library.db` must exist locally for step 1 to succeed. Steps 2 and 3 require internet access.

In [86]:
# ─── Initialize the pipeline ──────────────────────────────────────────────────
# Creates 'collected_data.db' and sets up logging
pipeline = DataCollectionPipeline()

# ─── Step 1: Collect from a local SQLite database ─────────────────────────────
# Reads books with a rating above 4 from a pre-existing library database
# (Requires 'library.db' to exist — created in Part 1 of the tutorial)
library_data = pipeline.collect_from_database(
    "SELECT * FROM books WHERE rating > 4",  # SQL query
    os.path.join('databases', 'library.db'),  # Path to source database
)

# ─── Step 2: Collect from the GitHub REST API ─────────────────────────────────
# Fetches metadata (stars, forks, description, etc.) for the pandas repository
github_data = pipeline.collect_from_api(
    'https://api.github.com/repos/pandas-dev/pandas'
    # No authentication needed for public repos (but rate-limited to 60 req/hr)
)

# ─── Step 3: Scrape structured data from a webpage ───────────────────────────
# CSS selectors map field names to HTML elements on the target page
book_data = pipeline.collect_from_web(
    'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html',
    {
        'title': 'h1',  # Book title in <h1>
        'price': '.price_color',  # Price in element with class 'price_color'
        'availability': '.availability',  # In-stock status
        'description': '#product_description ~ p',  # Sibling <p> after #product_description
    },
)

# ─── Step 4: Print collection statistics ─────────────────────────────────────
stats = pipeline.get_collection_stats()
print("\n=== Collection Statistics ===")
print(f"API Records:     {stats['api_records']}")
print(f"Scraped Records: {stats['scraped_records']}")
print("\nLogs by Source Type:")
print(stats['logs'])

# ─── Step 5: Export all collected data to CSV ─────────────────────────────────
# Creates: exports/api_data.csv, exports/scraped_data.csv, exports/pipeline_logs.csv
pipeline.export_all_data()

# ─── Step 6: Close the pipeline ───────────────────────────────────────────────
# Always close to release the database file lock
pipeline.close()

2026-03-08 18:38:08,047 - __main__ - INFO - Pipeline initialized
2026-03-08 18:38:08,054 - __main__ - INFO - Collecting from database: databases/library.db
2026-03-08 18:38:08,056 - __main__ - ERROR - ✗ Database error: Execution failed on sql 'SELECT * FROM books WHERE rating > 4': no such column: rating
2026-03-08 18:38:08,054 - __main__ - INFO - Collecting from database: databases/library.db
2026-03-08 18:38:08,056 - __main__ - ERROR - ✗ Database error: Execution failed on sql 'SELECT * FROM books WHERE rating > 4': no such column: rating
2026-03-08 18:38:08,064 - __main__ - INFO - Collecting from API: https://api.github.com/repos/pandas-dev/pandas
2026-03-08 18:38:08,064 - __main__ - INFO - Collecting from API: https://api.github.com/repos/pandas-dev/pandas


2026-03-08 18:38:08,568 - __main__ - INFO - Rate limit: 57/60 — resets at 2026-03-08 19:25:42
2026-03-08 18:38:08,578 - __main__ - INFO - ✓ Collected API data from https://api.github.com/repos/pandas-dev/pandas
2026-03-08 18:38:08,578 - __main__ - INFO - ✓ Collected API data from https://api.github.com/repos/pandas-dev/pandas
2026-03-08 18:38:08,585 - __main__ - INFO - Scraping: http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
2026-03-08 18:38:08,585 - __main__ - INFO - Scraping: http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
2026-03-08 18:38:09,486 - __main__ - INFO - ✓ Scraped data from http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
2026-03-08 18:38:09,486 - __main__ - INFO - ✓ Scraped data from http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
2026-03-08 18:38:09,510 - __main__ - INFO - ✓ Data exported to exports/
2026-03-08 18:38:09,512 - __main__ - INFO - Pipeline closed
2026-03-08 1


=== Collection Statistics ===
API Records:     1
Scraped Records: 1

Logs by Source Type:
  source_type  count  successful
0         api      1           1
1    database      1           0
2         web      1           1


---

## <span style="color:blue">4.2 Final Project: Book Market Intelligence System</span>

**Objective**: Create a **complete data collection system** that gathers book data from multiple sources and generates **market insights**.

---

### 📦 Data Sources *(all 3 required)*

| # | Source | Description |
|---|--------|-------------|
| 1 | **Database** | Use the library database from Part 1 |
| 2 | **API** | Use GitHub API to find book-related repositories |
| 3 | **Web** | Scrape the Books to Scrape website |

---

### 📋 Deliverables

**1. Code** (`final_project.py`):
- Complete `DataCollectionPipeline` implementation
- All three collection methods working
- Error handling and logging
- Data validation

**2. Database** (`market_intelligence.db`):
- Properly structured tables
- Collected data from all sources
- Relationships between tables

**3. Analysis Report** (`analysis.pdf` or `analysis.html`):
- Executive summary
- Data collection statistics
- Market insights:
  - Popular genres
  - Price trends
  - Rating patterns
  - Technology trends (from GitHub)
- Visualizations (**at least 5**)
- Recommendations

**4. Documentation** (`README.md`):
- Installation instructions
- How to run
- Architecture explanation
- Data schema diagram

---

### 🏆 Grading *(100 points)*

| Category | Points | Breakdown |
|---|---|---|
| **Data Collection** | 40 | Database (10) + API (15) + Web Scraping (15) |
| **Code Quality** | 25 | Clean code (10) + Error handling (8) + Logging (7) |
| **Analysis** | 25 | Insights quality (15) + Visualizations (10) |
| **Documentation** | 10 | README completeness (5) + Code comments (5) |


---

## <span style="color:blue">✅ Key Takeaways</span>

| Concept | Summary |
|---|---|
| 🗄️ **Databases** | Best for structured data, complex queries, and data integrity |
| 🌐 **APIs** | Official channels, real-time data — rate limits apply |
| 🕷️ **Web Scraping** | Last resort — be ethical, always check `robots.txt` |
| 🔗 **Integration** | Combine sources for comprehensive insights |
| ✅ **Best Practices** | Error handling, logging, validation, documentation |

---

### <span style="color:red">Remember:</span>

- Always **check if an API exists** before resorting to scraping
- **Respect rate limits** and `robots.txt`
- **Validate and clean** your data before analysis
- **Document your process** for reproducibility
- Be **ethical and legal** in all data collection

---

> 🎓 **End of Tutorial** — You now have a fully integrated, production-style data pipeline!

In [87]:
pipeline = DataCollectionPipeline('market_intelligence.db')

db_books = pipeline.collect_from_database(
    '''
    SELECT b.title, b.genre, b.publication_year, b.copies_available,
           a.name AS author
    FROM books b
    JOIN authors a ON b.author_id = a.author_id
    ''',
    os.path.join('databases', 'library.db'),
)
print(f"DB books: {len(db_books)} records")

gh_data = pipeline.collect_from_api(
    'https://api.github.com/search/repositories',
    params={'q': 'books language:python', 'sort': 'stars', 'per_page': 10},
)
if gh_data:
    gh_repos = pd.DataFrame([
        {
            'name':        r['name'],
            'full_name':   r['full_name'],
            'stars':       r['stargazers_count'],
            'forks':       r['forks_count'],
            'language':    r.get('language'),
            'description': r.get('description'),
        }
        for r in gh_data['items']
    ])
    print(f"GitHub repos: {len(gh_repos)} records")
else:
    gh_repos = pd.DataFrame()
    print("GitHub API unavailable")

rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

selectors = {
    'title':    'h3 a',
    'price':    '.price_color',
    'rating':   '.star-rating',
    'in_stock': '.availability',
}

cleaners = {
    'title':    lambda el: el['title'] if el else None,
    'price':    lambda el: float(el.get_text(strip=True).replace('\u00c2\u00a3', '').replace('\u00a3', '')) if el else None,
    'rating':   lambda el: rating_map.get(el['class'][1], 0) if el else 0,
    'in_stock': lambda el: 'In stock' in el.get_text() if el else False,
}

validators = {
    'title': lambda v: v is not None,           # must have a title
    'price': lambda v: v is not None and v > 0, # price must be a positive number
    'rating': lambda v: v is not None and 1 < v < 5, # rating must be between 1 and 5
    'in_stock': lambda v: v is not None
}

urls = [
    ('fiction',      'http://books.toscrape.com/catalogue/category/books/fiction_10/index.html'),
    ('autobiography', 'https://books.toscrape.com/catalogue/category/books/autobiography_27/index.html'),
]

web_books_results = []

for category, url in urls:
    books = pipeline.collect_from_web(
        url, selectors, cleaners=cleaners, validators=validators,
        container_selector='article.product_pod',
        next_page_selector='.next a', max_pages=2,
    )
    for book in books:
        book['category'] = category  
    web_books_results.extend(books)


web_books = pd.DataFrame(web_books_results)
print(f"Web books scraped: {len(web_books)} records")

stats = pipeline.get_collection_stats()
print("\n=== Collection Statistics ===")
print(f"API Records:     {stats['api_records']}")
print(f"Scraped Records: {stats['scraped_records']}")
print("\nLogs by Source Type:")
print(stats['logs'].to_string(index=False))

pipeline.export_all_data('exports')

pipeline.close()


2026-03-08 18:38:18,232 - __main__ - INFO - Pipeline initialized
2026-03-08 18:38:18,233 - __main__ - INFO - Collecting from database: databases/library.db
2026-03-08 18:38:18,235 - __main__ - INFO - ✓ Collected 10 records from database
2026-03-08 18:38:18,240 - __main__ - INFO - Collecting from API: https://api.github.com/search/repositories
2026-03-08 18:38:18,233 - __main__ - INFO - Collecting from database: databases/library.db
2026-03-08 18:38:18,235 - __main__ - INFO - ✓ Collected 10 records from database
2026-03-08 18:38:18,240 - __main__ - INFO - Collecting from API: https://api.github.com/search/repositories


DB books: 10 records


2026-03-08 18:38:19,200 - __main__ - INFO - Rate limit: 9/10 — resets at 2026-03-08 18:39:18
2026-03-08 18:38:19,201 - __main__ - WARNING - ⚠️ Low on API requests!
2026-03-08 18:38:19,207 - __main__ - INFO - ✓ Collected API data from https://api.github.com/search/repositories
2026-03-08 18:38:19,201 - __main__ - WARNING - ⚠️ Low on API requests!
2026-03-08 18:38:19,207 - __main__ - INFO - ✓ Collected API data from https://api.github.com/search/repositories
2026-03-08 18:38:19,213 - __main__ - INFO - Scraping: http://books.toscrape.com/catalogue/category/books/fiction_10/index.html
2026-03-08 18:38:19,213 - __main__ - INFO - Scraping: http://books.toscrape.com/catalogue/category/books/fiction_10/index.html


GitHub repos: 10 records


2026-03-08 18:38:20,546 - __main__ - INFO - ✓ Scraped 20 items from http://books.toscrape.com/catalogue/category/books/fiction_10/index.html (9 dropped by validators)
2026-03-08 18:38:20,555 - __main__ - INFO - Scraping: http://books.toscrape.com/catalogue/category/books/fiction_10/page-2.html
2026-03-08 18:38:20,555 - __main__ - INFO - Scraping: http://books.toscrape.com/catalogue/category/books/fiction_10/page-2.html
2026-03-08 18:38:21,494 - __main__ - INFO - ✓ Scraped 20 items from http://books.toscrape.com/catalogue/category/books/fiction_10/page-2.html (8 dropped by validators)
2026-03-08 18:38:21,510 - __main__ - INFO - Scraping: https://books.toscrape.com/catalogue/category/books/autobiography_27/index.html
2026-03-08 18:38:21,494 - __main__ - INFO - ✓ Scraped 20 items from http://books.toscrape.com/catalogue/category/books/fiction_10/page-2.html (8 dropped by validators)
2026-03-08 18:38:21,510 - __main__ - INFO - Scraping: https://books.toscrape.com/catalogue/category/books/a

Web books scraped: 27 records

=== Collection Statistics ===
API Records:     1
Scraped Records: 27

Logs by Source Type:
source_type  count  successful
        api      1           1
   database      1           1
        web      2           2


In [75]:
EXPORTS_DIR = "exports"
os.makedirs(EXPORTS_DIR, exist_ok=True)

In [118]:
# 1: DB — books per genre 
fig1, ax1 = plt.subplots(figsize=(7, 4))
if not db_books.empty:
    db_books['genre'].value_counts().plot(kind='bar', ax=ax1, color='steelblue')
    ax1.set_title('Library DB: Books per Genre')
    ax1.set_xlabel('Genre')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=30)
else:
    ax1.set_title('Library DB: No Data')

fig1.tight_layout()
fig1.savefig(f'{EXPORTS_DIR}/fig_db_genre_count.png', dpi=120, bbox_inches='tight')
plt.close(fig1)

In [117]:
# Figure 2: DB — copies available per genre 
fig2, ax2 = plt.subplots(figsize=(7, 4))
if not db_books.empty:
    db_books.groupby('genre')['copies_available'].sum().plot(
        kind='bar', ax=ax2, color='teal'
    )
    ax2.set_title('Library DB: Copies per Genre')
    ax2.set_xlabel('Genre')
    ax2.set_ylabel('Total Copies')
    ax2.tick_params(axis='x', rotation=30)
else:
    ax2.set_title('Library DB: No Data')
fig2.tight_layout()
fig2.savefig(f'{EXPORTS_DIR}/fig_db_genre_copies.png', dpi=120, bbox_inches='tight')
plt.close(fig2)

In [116]:
# Figure 3: GitHub — top repos by stars 
fig3, ax3 = plt.subplots(figsize=(7, 4))
if not gh_repos.empty:
    gh_repos.nlargest(10, 'stars').set_index('name')['stars'].plot(
        kind='barh', ax=ax3, color='orange'
    )
    ax3.set_title('GitHub: Top Book Repos by Stars')
    ax3.set_xlabel('Stars')
else:
    ax3.set_title('GitHub: No Data')
fig3.tight_layout()
fig3.savefig(f'{EXPORTS_DIR}/fig_github_stars.png', dpi=120, bbox_inches='tight')
plt.close(fig3)

In [115]:
# Figure 4: Web — price distribution per category 

cat_colors = {'fiction': 'red', 'autobiography': 'blue'}

if not web_books.empty:
    categories = list(web_books['category'].unique())
    fig4, axes4 = plt.subplots(1, len(categories), figsize=(7 * len(categories), 4), sharey=True)
    if len(categories) == 1:
        axes4 = [axes4]
    for ax, cat in zip(axes4, categories):
        grp = web_books[web_books['category'] == cat]
        grp['price'].plot(
            kind='hist', bins=10, ax=ax, alpha=0.8,
            color=cat_colors.get(cat, 'grey'), edgecolor='white'
        )
        ax.axvline(
            grp['price'].mean(), color=cat_colors.get(cat, 'grey'),
            linestyle='--', linewidth=1.5, label=f'mean £{grp["price"].mean():.2f}'
        )
        ax.set_title(f'{cat.capitalize()} ({len(grp)} books)')
        ax.set_xlabel('Price (£)')
        ax.set_ylabel('Count')
        ax.legend()
    fig4.suptitle('Web: Price Distribution by Category', fontsize=13, fontweight='bold')
else:
    fig4, ax4 = plt.subplots(figsize=(7, 4))
    ax4.set_title('Web: No Data')
fig4.tight_layout()
fig4.savefig(f'{EXPORTS_DIR}/fig_web_price_dist.png', dpi=120, bbox_inches='tight')
plt.close(fig4)

In [112]:
# Figure 5: Web — avg rating per category 

cat_colors = {'fiction': 'red', 'autobiography': 'blue'}

fig5, ax5 = plt.subplots(figsize=(7, 4))
if not web_books.empty:
    avg_rating = web_books.groupby('category')['rating'].mean()
    avg_rating.plot(
        kind='bar', ax=ax5,
        color=[cat_colors.get(c, 'grey') for c in avg_rating.index]
    )
    ax5.set_title('Web: Avg Rating by Category')
    ax5.set_xlabel('Category')
    ax5.set_ylabel('Avg Rating (stars)')
    ax5.set_ylim(0, 5)
    ax5.tick_params(axis='x', rotation=0)
    for bar, val in zip(ax5.patches, avg_rating):
        ax5.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9
        )
else:
    ax5.set_title('Web: No Data')
fig5.tight_layout()
fig5.savefig(f'{EXPORTS_DIR}/fig_web_avg_rating.png', dpi=120, bbox_inches='tight')
plt.close(fig5)

In [113]:
# Figure 6: Web — price vs rating scatter 

cat_colors = {'fiction': 'red', 'autobiography': 'blue'}

fig6, ax6 = plt.subplots(figsize=(7, 4))
if not web_books.empty:
    for cat, grp in web_books.groupby('category'):
        ax6.scatter(
            grp['rating'], grp['price'],
            color=cat_colors.get(cat, 'grey'),
            alpha=0.7, edgecolors='white', linewidths=0.5, label=cat
        )
    ax6.set_title('Web: Price vs Rating by Category')
    ax6.set_xlabel('Rating (stars)')
    ax6.set_ylabel('Price (\u00a3)')
    ax6.set_xticks([1, 2, 3, 4, 5])
    ax6.legend(title='Category')
else:
    ax6.set_title('Web: No Data')
fig6.tight_layout()
fig6.savefig(f'{EXPORTS_DIR}/fig_web_price_vs_rating.png', dpi=120, bbox_inches='tight')
plt.close(fig6)

In [114]:
print("=== Market Insights ===")
if not db_books.empty:
    print(f"Most common genre in library: {db_books['genre'].value_counts().idxmax()}")
if not gh_repos.empty:
    print(f"Top GitHub repo: {gh_repos.loc[gh_repos['stars'].idxmax(), 'full_name']} "
          f"({gh_repos['stars'].max():,} stars)")
if not web_books.empty:
    print(f"Web books scraped: {len(web_books)} total across {web_books['category'].nunique()} categories")
    for cat, grp in web_books.groupby('category'):
        print(f"  [{cat}] {len(grp)} books | avg price \u00a3{grp['price'].mean():.2f} | avg rating {grp['rating'].mean():.1f}\u2605")
    best_value = web_books.loc[web_books['price'].idxmin()]
    print(f"Cheapest book: '{best_value['title']}' at \u00a3{best_value['price']:.2f} ({best_value['category']}, rated {best_value['rating']}\u2605)")

=== Market Insights ===
Most common genre in library: Fiction
Top GitHub repo: EbookFoundation/free-programming-books (383,760 stars)
Web books scraped: 27 total across 2 categories
  [autobiography] 4 books | avg price £23.33 | avg rating 2.5★
  [fiction] 23 books | avg price £34.34 | avg rating 3.2★
Cheapest book: 'I Am Pilgrim (Pilgrim #1)' at £10.60 (fiction, rated 4★)
